# Pythia-160M SAE: trening albo analiza

Ten notebook jest przeznaczony do uruchamiania w Kaggle. Wykonuje dokładnie jeden z dwóch głównych etapów: `TRAIN` albo `ANALYZE`. Nie zawiera treningu pilotowego. Domyślne artefakty obejmują próbkę 50 mln tokenów przygotowaną z pliku `00.jsonl` (jeden shard datasetu The Pile), a nie cały wieloshardowy korpus The Pile. Jeśli przygotowane sekwencje nie są dostępne, opcjonalny fallback sekwencjonuje wyłącznie wskazany plik do `/kaggle/working/pythia160m_sae_pilecc/sequenced/`.

In [ ]:
import os
from pathlib import Path
import numpy as np
import shlex
import shutil
import subprocess
import sys

# =========================
# JEDYNE USTAWIENIA SESJI
# =========================
RUN_MODE = "ANALYZE"  # "TRAIN" albo "ANALYZE"
VERBOSE = "low"       # "low" ogranicza logi Kaggle; "high" pokazuje tqdm
VERBOSE_INTERVAL = 100_000          # kroki treningu albo chunki analizy
SEQUENCE_VERBOSE_INTERVAL = 100_000_000  # tokeny podczas sekwencjonowania

# None oznacza pełną analizę wszystkich przygotowanych sekwencji.
# Wartość całkowita uruchamia analizę tylko na próbce.
ANALYSIS_MAX_SEQUENCES = None
ANALYSIS_SAMPLING = "uniform"  # używane tylko, gdy ustawiono limit
RESUME_ANALYSIS = True
CHECKPOINT_EVERY_CHUNKS = 10
FINALIZE_CHECKPOINT_ONLY = False  # True: raport z checkpointu bez dalszej analizy
REQUIRE_CHECKPOINT_INPUT = True  # False tylko dla pierwszego runu bez checkpointu
USE_PREPARED_SEQUENCES = True  # False włącza fallback dla pojedynczego sharda JSONL

def resolve_kaggle_dataset_dir(owner, slug):
    candidates = [
        Path('/kaggle/input/datasets') / owner / slug,
        Path('/kaggle/input') / slug,
    ]
    matches = [path for path in candidates if path.is_dir()]
    if not matches:
        raise FileNotFoundError(
            f'Dataset {owner}/{slug} nie jest podpięty; sprawdzono: {candidates}'
        )
    return matches[0]

CODE_DIR = resolve_kaggle_dataset_dir('erykmikoajek', 'sae-training-and-moeffication')
os.environ['PYTHONPATH'] = str(CODE_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
PREPARED_SEQUENCE_DIR = resolve_kaggle_dataset_dir('erykmikoajek', 'trained-sae-models')
MAIN = CODE_DIR / 'sae_pipeline' / 'main.py'
RESULTS_DIR = Path("/kaggle/working/pythia160m_sae_pilecc")
PILE_DATASET_DIR = None
if not USE_PREPARED_SEQUENCES:
    PILE_DATASET_DIR = resolve_kaggle_dataset_dir('dschettler8845', 'the-pile-dataset-part-00-of-29')
PILE_JSONL = PILE_DATASET_DIR / '00.jsonl' if PILE_DATASET_DIR is not None else None
BEST_SAE = PREPARED_SEQUENCE_DIR / 'topk_sae_layer_6_best_pilecc.pt'
MODEL_NAME = "EleutherAI/pythia-160m"

LAYER_NUM = 6
SEQ_LENGTH = 256
MODEL_BATCH_SIZE = 2
CHUNK_SEQUENCES = 16
BATCH_SIZE_SAE = 1024
EXPANSION_FACTOR = 16
K = 64
NUM_EPOCHS = 2
TRAIN_RESUME = True
# Gotowe sekwencje z Kaggle Input omijają ponowne przetwarzanie JSONL.
PREPARED_TOKENS_PATH = PREPARED_SEQUENCE_DIR / 'tokens_seqs_padded_pythia.npy'
PREPARED_MASK_PATH = PREPARED_SEQUENCE_DIR / 'attention_mask_pythia.npy'
# Opcjonalnie: checkpoint zapisany wcześniej jako plik Kaggle Dataset.
# Po skopiowaniu do working analizator będzie go aktualizował lokalnie.
CHECKPOINT_INPUT_PATH = PREPARED_SEQUENCE_DIR / 'analysis_checkpoint_pythia.pt'
CHECKPOINT_WORKING_PATH = RESULTS_DIR / 'analysis' / 'analysis_checkpoint_pythia.pt'
AUTO_SEQUENCE_IF_MISSING = False
SEQUENCE_MAX_TOKENS = None  # None = cały plik; np. 10_000_000 = limit testowy
SEQUENCE_MAX_DOCUMENTS = None

if RUN_MODE not in {"TRAIN", "ANALYZE"}:
    raise ValueError("RUN_MODE musi mieć wartość 'TRAIN' albo 'ANALYZE'")
if VERBOSE not in {"low", "high"}:
    raise ValueError("VERBOSE musi mieć wartość 'low' albo 'high'")
if VERBOSE_INTERVAL < 1:
    raise ValueError("VERBOSE_INTERVAL musi być dodatni")
if CHECKPOINT_EVERY_CHUNKS < 1:
    raise ValueError("CHECKPOINT_EVERY_CHUNKS musi być dodatni")
if FINALIZE_CHECKPOINT_ONLY and RUN_MODE != 'ANALYZE':
    raise ValueError("FINALIZE_CHECKPOINT_ONLY dotyczy wyłącznie RUN_MODE='ANALYZE'")
if min(SEQ_LENGTH, MODEL_BATCH_SIZE, CHUNK_SEQUENCES, BATCH_SIZE_SAE) < 1:
    raise ValueError("Rozmiary sekwencji, batchy i chunków muszą być dodatnie")
if SEQUENCE_VERBOSE_INTERVAL < 1:
    raise ValueError("SEQUENCE_VERBOSE_INTERVAL musi być dodatni")
if ANALYSIS_MAX_SEQUENCES is not None and ANALYSIS_MAX_SEQUENCES < 1:
    raise ValueError("ANALYSIS_MAX_SEQUENCES musi być dodatni albo None")
if ANALYSIS_SAMPLING not in {"head", "tail", "uniform"}:
    raise ValueError("ANALYSIS_SAMPLING musi być: head, tail albo uniform")
if SEQUENCE_MAX_TOKENS is not None and SEQUENCE_MAX_TOKENS < 1:
    raise ValueError("SEQUENCE_MAX_TOKENS musi być dodatni albo None")
if SEQUENCE_MAX_DOCUMENTS is not None and SEQUENCE_MAX_DOCUMENTS < 1:
    raise ValueError("SEQUENCE_MAX_DOCUMENTS musi być dodatni albo None")
if not CODE_DIR.is_dir():
    raise FileNotFoundError(f"Brak katalogu ze źródłami: {CODE_DIR}")
if not MAIN.is_file():
    raise FileNotFoundError(f"Brak pliku main.py: {MAIN}")
required_sources = ['sae_pipeline/autoencoder_training.py', 'sae_pipeline/activations_collecting.py', 'sae_pipeline/features_analysis.py', 'sae_pipeline/dataset_sequencing.py', 'common/utils.py']
missing_sources = [name for name in required_sources if not (CODE_DIR / name).is_file()]
if missing_sources:
    raise FileNotFoundError(f"Brak plików źródłowych w Kaggle dataset: {missing_sources}")
main_source = MAIN.read_text(encoding="utf-8")
if "os." in main_source and "import os" not in main_source:
    raise RuntimeError("main.py używa modułu os, ale go nie importuje. Zaktualizuj Kaggle dataset ze źródłami.")
source_markers = {
    'sae_pipeline/main.py': ["--verbose-interval", "--tokens-path", "--attention-mask-path", "--require-prepared-sequences", "--analysis-checkpoint-path", "--no-analysis-resume", "--analysis-checkpoint-every-chunks"],
    'sae_pipeline/autoencoder_training.py': ["verbose_interval"],
    'sae_pipeline/features_analysis.py': ["verbose_interval", "sequence-aware-v1", "migrate_flattened_context_examples"],
    'sae_pipeline/dataset_sequencing.py': ["root.text", "verbose_interval"],
}
for source_name, markers in source_markers.items():
    source_text = (CODE_DIR / source_name).read_text(encoding="utf-8")
    missing_markers = [marker for marker in markers if marker not in source_text]
    if missing_markers:
        raise RuntimeError(
            f"Nieaktualny {source_name}; brak elementów {missing_markers}. "
            "Ponownie opublikuj zaktualizowany dataset ze źródłami."
        )

print(f"RUN_MODE: {RUN_MODE}")
print(f"VERBOSE: {VERBOSE}, interval={VERBOSE_INTERVAL}")
print(f"Sequence verbose interval: {SEQUENCE_VERBOSE_INTERVAL:,} tokens")
print(f"Results/sequences: {RESULTS_DIR}")
print(f"Checkpoint: {BEST_SAE}")
print(f"Raw Pile shard (fallback only): {PILE_JSONL or '<not mounted>'}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', '<not set>')}")

In [ ]:
def run_cli(*args):
    command = [sys.executable, str(MAIN), *map(str, args)]
    print(shlex.join(command))
    subprocess.run(command, check=True)

if RUN_MODE == "ANALYZE" and CHECKPOINT_INPUT_PATH is not None:
    if CHECKPOINT_INPUT_PATH.is_file():
        if not CHECKPOINT_WORKING_PATH.is_file():
            CHECKPOINT_WORKING_PATH.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(CHECKPOINT_INPUT_PATH, CHECKPOINT_WORKING_PATH)
            print(f"Przywrócono checkpoint do working: {CHECKPOINT_WORKING_PATH}")
    elif REQUIRE_CHECKPOINT_INPUT or FINALIZE_CHECKPOINT_ONLY:
        raise FileNotFoundError(f"Brak wymaganego checkpointu wejściowego: {CHECKPOINT_INPUT_PATH}")
    else:
        print("Brak checkpointu wejściowego: analiza rozpocznie się od początku.")

COMMON_ARGS = [
    "--profile", "local-50gb",
    "--model-name", MODEL_NAME,
    "--tokenizer-name", MODEL_NAME,
    "--data-path", str(RESULTS_DIR),
    "--layer-num", str(LAYER_NUM),
    "--seq-length", str(SEQ_LENGTH),
    "--model-batch-size", str(MODEL_BATCH_SIZE),
    "--chunk-sequences", str(CHUNK_SEQUENCES),
    "--batch-size-sae", str(BATCH_SIZE_SAE),
    "--expansion-factor", str(EXPANSION_FACTOR),
    "--k", str(K),
    "--num-epochs", str(NUM_EPOCHS),
    "--verbose", VERBOSE,
    "--verbose-interval", str(VERBOSE_INTERVAL),
    "--no-interactive",
]
if USE_PREPARED_SEQUENCES and not FINALIZE_CHECKPOINT_ONLY:
    COMMON_ARGS.extend([
        "--tokens-path", str(PREPARED_TOKENS_PATH),
        "--attention-mask-path", str(PREPARED_MASK_PATH),
        "--require-prepared-sequences",
    ])

if USE_PREPARED_SEQUENCES and not FINALIZE_CHECKPOINT_ONLY:
    for path, label in ((PREPARED_TOKENS_PATH, "sekwencje"), (PREPARED_MASK_PATH, "maska atencji")):
        if not path.is_file():
            raise FileNotFoundError(f"Brak gotowych {label}: {path}")
    token_shape = np.load(PREPARED_TOKENS_PATH, mmap_mode='r').shape
    mask_shape = np.load(PREPARED_MASK_PATH, mmap_mode='r').shape
    if len(token_shape) != 2 or token_shape != mask_shape or token_shape[1] != SEQ_LENGTH:
        raise ValueError(
            f'Niezgodne przygotowane sekwencje: tokens={token_shape}, mask={mask_shape}, '
            f'oczekiwana długość={SEQ_LENGTH}'
        )
    print(f"Gotowe sekwencje: {token_shape}; maska: {mask_shape}.")
elif not FINALIZE_CHECKPOINT_ONLY:
    sequence_tokens = RESULTS_DIR / "sequenced" / "tokens_seqs_padded.npy"
    sequence_mask = RESULTS_DIR / "sequenced" / "attention_mask.npy"
    missing_sequence_files = [
        path for path in (sequence_tokens, sequence_mask) if not path.is_file()
    ]
    if missing_sequence_files:
        print("Brak przygotowanych sekwencji w RESULTS_DIR:")
        for path in missing_sequence_files:
            print(f"  - {path}")
        if not AUTO_SEQUENCE_IF_MISSING:
            raise FileNotFoundError(
                "AUTO_SEQUENCE_IF_MISSING=False, więc sekwencjonowanie nie zostanie uruchomione."
            )
        if PILE_JSONL is None or not PILE_JSONL.is_file():
            raise FileNotFoundError(f"Nie znaleziono źródłowego datasetu The Pile: {PILE_JSONL}")
        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        sequence_args = [
            "--stage", "sequence",
            "--profile", "local-50gb",
            "--model-name", MODEL_NAME,
            "--tokenizer-name", MODEL_NAME,
            "--data-path", str(RESULTS_DIR),
            "--input-path", str(PILE_JSONL),
            "--seq-length", str(SEQ_LENGTH),
            "--min-seq-length", "10",
            "--batch-sentences", "32",
            "--verbose", VERBOSE,
            "--verbose-interval", str(SEQUENCE_VERBOSE_INTERVAL),
            "--no-interactive",
        ]
        if SEQUENCE_MAX_TOKENS is not None:
            sequence_args.extend(["--max-tokens", str(SEQUENCE_MAX_TOKENS)])
        if SEQUENCE_MAX_DOCUMENTS is not None:
            sequence_args.extend(["--max-documents", str(SEQUENCE_MAX_DOCUMENTS)])
        print("Uruchamiam automatyczne sekwencjonowanie The Pile...")
        run_cli(*sequence_args)
    if not sequence_tokens.is_file() or not sequence_mask.is_file():
        raise RuntimeError(f"Brak sekwencji po przygotowaniu: {sequence_tokens}, {sequence_mask}")
    print(f"Prepared tokens: {sequence_tokens}")
    print(f"Attention mask: {sequence_mask}")
else:
    print("FINALIZE_CHECKPOINT_ONLY=True: sekwencje wejściowe nie są potrzebne.")

## Trening SAE

Uruchomi się tylko wtedy, gdy `RUN_MODE = "TRAIN"`. Wznowienie korzysta z checkpointu znajdującego się w `RESULTS_DIR/models/checkpoints/`; zmiana `VERBOSE` nie zmienia konfiguracji modelu.

In [ ]:
if RUN_MODE == "TRAIN":
    train_args = ["--stage", "train-sae", *COMMON_ARGS]
    if not TRAIN_RESUME:
        train_args.append("--no-resume")
    run_cli(*train_args)
else:
    print("Pomijam trening: RUN_MODE=ANALYZE")

## Pełna analiza SAE

Domyślnie `ANALYSIS_MAX_SEQUENCES = None`, więc analizator przechodzi po całym przygotowanym `tokens_seqs_padded_pythia.npy` (50 mln tokenów z sharda `00.jsonl`, nie po całym korpusie The Pile). Dzięki temu można porównać cechy obserwowane w analizie z `usage_counts` zapisanym podczas treningu i odróżnić cechy martwe od niewidzianych w danej próbce. `FINALIZE_CHECKPOINT_ONLY = True` tworzy raport wyłącznie z obecnego `analysis_checkpoint_pythia.pt`, bez uruchamiania kolejnych batchy.

In [ ]:
if RUN_MODE == "ANALYZE":
    if not BEST_SAE.is_file():
        raise FileNotFoundError(f"Brak checkpointu SAE: {BEST_SAE}")
    analyze_args = [
        "--stage", "finalize-analysis" if FINALIZE_CHECKPOINT_ONLY else "analyze",
        *COMMON_ARGS,
        "--checkpoint-path", str(BEST_SAE),
    ]
    if ANALYSIS_MAX_SEQUENCES is not None:
        analyze_args.extend([
            "--max-sequences", str(ANALYSIS_MAX_SEQUENCES),
            "--analysis-sampling", ANALYSIS_SAMPLING,
        ])
    analyze_args.extend(["--analysis-checkpoint-every-chunks", str(CHECKPOINT_EVERY_CHUNKS)])
    analyze_args.extend(["--analysis-checkpoint-path", str(CHECKPOINT_WORKING_PATH)])
    if not RESUME_ANALYSIS:
        analyze_args.append("--no-analysis-resume")
    run_cli(*analyze_args)
else:
    print("Pomijam analizę: RUN_MODE=TRAIN")

In [ ]:
if RUN_MODE == 'ANALYZE':
    if not CHECKPOINT_WORKING_PATH.is_file():
        raise RuntimeError(f'Analiza nie zapisała checkpointu: {CHECKPOINT_WORKING_PATH}')
    print(f'Checkpoint output: {CHECKPOINT_WORKING_PATH} ({CHECKPOINT_WORKING_PATH.stat().st_size:,} B)')
print("Gotowe. Wyniki i logi analizy/treningu są w: " + str(RESULTS_DIR))
print("Analiza tekstowa: " + str(RESULTS_DIR / 'analysis' / 'features_analysis.txt'))
print("Analiza JSON: " + str(RESULTS_DIR / 'analysis' / 'features_analysis.json'))